# Images et vidéos avec OpenCV et scikit-image

Dans ce TP, vous allez détecter un visage dans une image, puis suivre un cheval dans une vidéo de polo. scikit-image fournit les algorithmes de traitement d'images, OpenCV lit et écrit les vidéos, image par image.

1. Détecter des visages
2. Lire une vidéo image par image
3. Suivre le cheval par corrélation
4. Écrire la vidéo annotée
5. Pour aller plus loin : mettre à jour le modèle
6. Pour aller plus loin : un algorithme de suivi d'OpenCV

Les exercices 2 à 6 s'enchaînent : chacun se sert de ce que les solutions des précédents définissent. Chaque exercice a une cellule pour votre code, puis des conseils supplémentaires et sa solution, repliés : ouvrez-les si besoin.

Commencez par exécuter la cellule suivante, sans la lire : elle télécharge la vidéo et définit la fonction `afficher_video`.

In [ ]:
# @title À exécuter : mise en place de l'environnement
# La séquence « polo » du VOT Challenge : 111 images d'un match de polo.
![ -f polo.mp4 ] || (curl -fsSL -o polo.zip https://data.votchallenge.net/sequences/acc3aeade6fb699adb7ca1c28866dd00f06d8fcd02c4cff17539a3c9d718d38976f637f4499a8763ae65f93d71fcd37494747bec7bbb8955a469b9f3fd9783aa.zip && unzip -qo polo.zip -d polo)

import os
import pathlib
import shutil

import cv2
from IPython.display import Video, display

# Les images, réduites en 640 × 360, deviennent la vidéo polo.mp4, à 10 images/s.
if not pathlib.Path("polo.mp4").exists():
  writer = cv2.VideoWriter("polo.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 10, (640, 360))
  for path in sorted(pathlib.Path("polo").glob("*.jpg")):
    frame = cv2.resize(cv2.imread(str(path)), (640, 360), interpolation=cv2.INTER_AREA)
    writer.write(frame)
  writer.release()
  shutil.rmtree("polo")
  pathlib.Path("polo.zip").unlink()


def afficher_video(chemin: str) -> None:
  """Affiche une vidéo dans le notebook, réencodée en WebM pour le navigateur."""
  capture = cv2.VideoCapture(chemin)
  fps = capture.get(cv2.CAP_PROP_FPS)
  ok, frame = capture.read()
  height, width = frame.shape[:2]
  # OpenCV prévient que le format WebM n'a pas d'étiquette de codec, puis écrit
  # du VP8 quand même : son avertissement est masqué.
  stderr = os.dup(2)
  devnull = os.open(os.devnull, os.O_WRONLY)
  os.dup2(devnull, 2)
  writer = cv2.VideoWriter(
    "displayed_video.webm", cv2.VideoWriter_fourcc(*"VP80"), fps, (width, height)
  )
  os.dup2(stderr, 2)
  os.close(devnull)
  os.close(stderr)
  while ok:
    writer.write(frame)
    ok, frame = capture.read()
  capture.release()
  writer.release()
  display(Video("displayed_video.webm", embed=True, mimetype="video/webm", width=640))

## Imports

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import patches
from skimage import color, data, feature

## 1. Détecter des visages

Un détecteur en cascade (la méthode de Viola et Jones) fait glisser une fenêtre sur l'image, à plusieurs échelles, et soumet chaque position à une suite de classifieurs très simples : la plupart des fenêtres sont rejetées par les premiers, et seules celles qui passent toute la cascade sont gardées comme visages. scikit-image fournit un tel détecteur, `feature.Cascade`, et les paramètres d'un modèle entraîné sur des visages vus de face.

Détectez le visage de l'image `data.astronaut()`, une photo de l'astronaute Eileen Collins :

- créez le détecteur à partir du fichier de paramètres dont `data.lbp_frontal_face_cascade_filename()` donne le chemin ;
- appelez sa méthode `detect_multi_scale` sur l'image, avec une taille minimale et une taille maximale des visages cherchés ;
- affichez l'image avec matplotlib, chaque visage détecté entouré d'un rectangle (`patches.Rectangle`).

Élargissez ensuite la plage des tailles cherchées, de 20 à 300 pixels : que se passe-t-il ?

**Documentation utile**

- https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.Cascade
- https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.Cascade.detect_multi_scale
- https://scikit-image.org/docs/stable/api/skimage.data.html#skimage.data.lbp_frontal_face_cascade_filename
- https://matplotlib.org/stable/api/_as_gen/matplotlib.patches.Rectangle.html

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `feature.Cascade(chemin)` charge le modèle.
- `detect_multi_scale(img=image, scale_factor=1.2, step_ratio=1, min_size=(60, 60), max_size=(123, 123))` : `scale_factor` est le rapport entre deux échelles successives de la fenêtre, `step_ratio` le pas de son glissement (`1` : chaque pixel), `min_size` et `max_size` les tailles (hauteur, largeur) des visages cherchés, en pixels.
- Chaque détection est un dictionnaire : `r` et `c` sont la ligne et la colonne de son coin en haut à gauche, `width` et `height` sa taille. Avec les paramètres ci-dessus, vous devez trouver un seul visage, de 90 × 90 pixels, en `r=69, c=177`.
- `patches.Rectangle((x, y), largeur, hauteur, fill=False, edgecolor="red")` attend l'abscisse `x`, la colonne, avant l'ordonnée `y`, la ligne ; `ax.add_patch(rectangle)` l'ajoute à la figure.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
detecteur = feature.Cascade(...)
astronaute = data.astronaut()
detections = detecteur.detect_multi_scale(
  img=astronaute, scale_factor=1.2, step_ratio=1, min_size=..., max_size=...
)

fig, ax = plt.subplots()
ax.imshow(astronaute)
for detection in detections:
  rectangle = patches.Rectangle(
    (..., ...),
    detection["width"],
    detection["height"],
    fill=False,
    edgecolor="red",
    linewidth=2,
  )
  ax.add_patch(rectangle)
ax.axis("off")
plt.show()
```

### Solution

In [ ]:
detecteur = feature.Cascade(data.lbp_frontal_face_cascade_filename())
astronaute = data.astronaut()


def afficher_visages(
  image: np.ndarray, min_size: tuple[int, int], max_size: tuple[int, int]
) -> None:
  detections = detecteur.detect_multi_scale(
    img=image, scale_factor=1.2, step_ratio=1, min_size=min_size, max_size=max_size
  )
  print(f"{len(detections)} visage(s) détecté(s)")
  fig, ax = plt.subplots()
  ax.imshow(image)
  for detection in detections:
    rectangle = patches.Rectangle(
      (detection["c"], detection["r"]),
      detection["width"],
      detection["height"],
      fill=False,
      edgecolor="red",
      linewidth=2,
    )
    ax.add_patch(rectangle)
  ax.axis("off")
  plt.show()


afficher_visages(astronaute, (60, 60), (123, 123))
# Une plage de tailles plus large teste plus de fenêtres : le calcul est plus
# long, et le détecteur prend d'autres motifs pour des visages.
afficher_visages(astronaute, (20, 20), (300, 300))

## Une vidéo de polo

La cellule de mise en place a créé `polo.mp4` : les 111 images d'un match de polo de la séquence `polo` du VOT Challenge, une compétition d'algorithmes de suivi d'objets, réduites en 640 × 360 pixels, à 10 images par seconde. Elle a aussi défini `afficher_video(chemin)`, qui affiche une vidéo dans le notebook. Regardez la vidéo : vous allez suivre le cheval de gauche.

In [ ]:
afficher_video("polo.mp4")

## 2. Lire une vidéo image par image

Écrivez la fonction `lire_images(chemin)`, qui lit une vidéo avec `cv2.VideoCapture`, comme dans le cours, et renvoie la liste de ses images, chacune convertie de BGR, l'ordre des canaux d'OpenCV, en RGB, celui de matplotlib et de scikit-image.

Lisez `polo.mp4` dans `images`, affichez le nombre d'images et la forme d'une image, puis affichez la première avec matplotlib, avec le rectangle `CADRE_CHEVAL` : c'est le cadre du cheval à suivre. Un cadre est un tuple `(x, y, largeur, hauteur)`, `(x, y)` étant son coin en haut à gauche.

In [ ]:
CADRE_CHEVAL = (160, 160, 80, 73)  # (x, y, largeur, hauteur)


def lire_images(chemin: str) -> list[np.ndarray]:
  # Votre code ici
  ...

### Conseils supplémentaires

- `capture.read()` renvoie un couple `(ok, image)` : `ok` vaut `False` une fois la vidéo terminée, et l'image est alors `None`.
- `cv2.cvtColor(image, cv2.COLOR_BGR2RGB)` convertit une image de BGR en RGB.
- Vous devez obtenir 111 images de forme `(360, 640, 3)`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
CADRE_CHEVAL = (160, 160, 80, 73)


def lire_images(chemin: str) -> list[np.ndarray]:
  capture = cv2.VideoCapture(chemin)
  images = []
  ok, image = capture.read()
  while ok:
    images.append(...)
    ok, image = capture.read()
  capture.release()
  return images


images = lire_images("polo.mp4")
print(len(images), images[0].shape)

x, y, largeur, hauteur = CADRE_CHEVAL
fig, ax = plt.subplots()
ax.imshow(...)
ax.add_patch(patches.Rectangle(...))
plt.show()
```

### Solution

In [ ]:
CADRE_CHEVAL = (160, 160, 80, 73)  # (x, y, largeur, hauteur)


def lire_images(chemin: str) -> list[np.ndarray]:
  capture = cv2.VideoCapture(chemin)
  images = []
  ok, image = capture.read()
  while ok:
    # OpenCV lit les images en BGR, matplotlib et scikit-image les veulent en RGB.
    images.append(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ok, image = capture.read()
  capture.release()
  return images


images = lire_images("polo.mp4")
print(len(images), images[0].shape)

x, y, largeur, hauteur = CADRE_CHEVAL
fig, ax = plt.subplots()
ax.imshow(images[0])
ax.add_patch(
  patches.Rectangle((x, y), largeur, hauteur, fill=False, edgecolor="red", linewidth=2)
)
plt.show()

## 3. Suivre le cheval par corrélation

Le suivi le plus simple cherche un modèle, l'imagette du cheval dans la première image, dans chacune des images suivantes. `feature.match_template(zone, modele)` calcule, pour chaque position du modèle dans une zone de l'image, la corrélation normalisée entre le modèle et l'image, entre -1 et 1 : son maximum indique où l'image ressemble le plus au modèle.

Écrivez la fonction `suivre(images, cadre, marge=30)`, qui renvoie le cadre du cheval dans chaque image :

- le modèle est le contenu de `cadre` dans la première image, en niveaux de gris ;
- dans chaque image suivante, cherchez le modèle dans une zone qui déborde du cadre précédent de `marge` pixels de chaque côté, et placez le nouveau cadre à la position du maximum de corrélation.

Suivez le cheval à partir de `CADRE_CHEVAL` et affichez un cadre sur dix. Essayez ensuite une marge de 10 pixels, puis une marge qui couvre toute l'image (1000 pixels) : que se passe-t-il ? Mesurez le temps de calcul avec `%time`.

**Documentation utile**

- https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.match_template
- https://scikit-image.org/docs/stable/api/skimage.color.html#skimage.color.rgb2gray
- https://numpy.org/doc/stable/reference/generated/numpy.unravel_index.html

In [ ]:
def suivre(
  images: list[np.ndarray], cadre: tuple[int, int, int, int], marge: int = 30
) -> list[tuple[int, int, int, int]]:
  # Votre code ici
  ...

### Conseils supplémentaires

- `color.rgb2gray(image)` convertit une image RGB en niveaux de gris, des flottants entre 0 et 1. Le modèle est `image[y : y + hauteur, x : x + largeur]` : les lignes d'abord, puis les colonnes.
- La zone commence en `(x0, y0) = (max(x - marge, 0), max(y - marge, 0))`, pour ne pas sortir de l'image, et s'arrête `marge` pixels après le cadre (une tranche NumPy qui dépasse s'arrête au bord de l'image).
- `feature.match_template(zone, modele)` renvoie un tableau de corrélations, une par position du coin en haut à gauche du modèle dans la zone. `np.unravel_index(np.argmax(scores), scores.shape)` donne la ligne et la colonne `(dy, dx)` de son maximum : le nouveau cadre commence en `(x0 + dx, y0 + dy)`.
- Avec une marge de 30 pixels, le dernier cadre est proche de `(189, 191, 80, 73)`.
- `%time` devant une ligne affiche le temps qu'elle prend.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def suivre(
  images: list[np.ndarray], cadre: tuple[int, int, int, int], marge: int = 30
) -> list[tuple[int, int, int, int]]:
  x, y, largeur, hauteur = cadre
  modele = color.rgb2gray(...)
  cadres = [cadre]
  for image in images[1:]:
    x0, y0 = max(x - marge, 0), max(y - marge, 0)
    zone = color.rgb2gray(image[y0 : y + hauteur + marge, x0 : x + largeur + marge])
    scores = feature.match_template(zone, modele)
    dy, dx = ...
    x, y = int(x0 + dx), int(y0 + dy)
    cadres.append((x, y, largeur, hauteur))
  return cadres


%time cadres = suivre(images, CADRE_CHEVAL)
print(cadres[::10])
print(suivre(images, CADRE_CHEVAL, marge=10)[::10])
%time print(suivre(images, CADRE_CHEVAL, marge=1000)[::10])
```

### Solution

In [ ]:
def suivre(
  images: list[np.ndarray], cadre: tuple[int, int, int, int], marge: int = 30
) -> list[tuple[int, int, int, int]]:
  x, y, largeur, hauteur = cadre
  modele = color.rgb2gray(images[0][y : y + hauteur, x : x + largeur])
  cadres = [cadre]
  for image in images[1:]:
    # La zone de recherche : le cadre précédent, élargi de la marge.
    x0, y0 = max(x - marge, 0), max(y - marge, 0)
    zone = color.rgb2gray(image[y0 : y + hauteur + marge, x0 : x + largeur + marge])
    scores = feature.match_template(zone, modele)
    dy, dx = np.unravel_index(np.argmax(scores), scores.shape)
    x, y = int(x0 + dx), int(y0 + dy)
    cadres.append((x, y, largeur, hauteur))
  return cadres


%time cadres = suivre(images, CADRE_CHEVAL)
print(cadres[::10])
# Une marge de 10 pixels perd le cheval, qui se déplace plus vite que cela.
print(suivre(images, CADRE_CHEVAL, marge=10)[::10])
# Une marge qui couvre toute l'image est plus lente, et le cadre finit par sauter
# sur l'autre cheval, qui ressemble aussi au modèle.
%time print(suivre(images, CADRE_CHEVAL, marge=1000)[::10])

## 4. Écrire la vidéo annotée

`cv2.VideoWriter` écrit une vidéo image par image, comme `cv2.VideoCapture` la lit. Écrivez la fonction `ecrire_video(images, cadres, chemin, fps=10)`, qui écrit dans `chemin` la vidéo des images, chacune avec son cadre dessiné par `cv2.rectangle`. Écrivez `polo_suivi.mp4` avec les cadres de l'exercice précédent, puis affichez-la avec `afficher_video`.

Deux pièges : OpenCV écrit des images en BGR, et `cv2.rectangle` dessine dans l'image qu'on lui passe. Dessinez dans une copie, pour garder les images de la liste intactes.

**Documentation utile**

- https://docs.opencv.org/4.x/dd/d9e/classcv_1_1VideoWriter.html
- https://docs.opencv.org/4.x/d6/d6e/group__imgproc__draw.html#ga07d2f74cadcf8e305e810ce8eed13bc9

In [ ]:
def ecrire_video(
  images: list[np.ndarray],
  cadres: list[tuple[int, int, int, int]],
  chemin: str,
  fps: int = 10,
) -> None:
  # Votre code ici
  ...

### Conseils supplémentaires

- `cv2.VideoWriter(chemin, cv2.VideoWriter_fourcc(*"mp4v"), fps, (largeur, hauteur))` crée la vidéo : sa taille est `(largeur, hauteur)`, dans l'ordre inverse de `image.shape`. Le FourCC (*four character code*) désigne le codec en quatre caractères, ici `mp4v`, du MPEG-4 ; les autres codes : https://fourcc.org/codecs.php.
- `writer.write(image)` ajoute une image, en BGR, et `writer.release()` termine le fichier.
- `cv2.rectangle(image, (x, y), (x + largeur, y + hauteur), color=(255, 0, 0), thickness=2)` prend deux coins opposés ; la couleur suit l'ordre des canaux de l'image : `(255, 0, 0)` est du rouge dans une image RGB.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def ecrire_video(
  images: list[np.ndarray],
  cadres: list[tuple[int, int, int, int]],
  chemin: str,
  fps: int = 10,
) -> None:
  hauteur_image, largeur_image = images[0].shape[:2]
  writer = cv2.VideoWriter(chemin, cv2.VideoWriter_fourcc(*"mp4v"), fps, (..., ...))
  for image, (x, y, largeur, hauteur) in zip(images, cadres, strict=True):
    annotee = image.copy()
    cv2.rectangle(annotee, ..., ..., color=(255, 0, 0), thickness=2)
    writer.write(cv2.cvtColor(annotee, ...))
  writer.release()


ecrire_video(images, cadres, "polo_suivi.mp4")
afficher_video("polo_suivi.mp4")
```

### Solution

In [ ]:
def ecrire_video(
  images: list[np.ndarray],
  cadres: list[tuple[int, int, int, int]],
  chemin: str,
  fps: int = 10,
) -> None:
  hauteur_image, largeur_image = images[0].shape[:2]
  writer = cv2.VideoWriter(
    chemin, cv2.VideoWriter_fourcc(*"mp4v"), fps, (largeur_image, hauteur_image)
  )
  for image, (x, y, largeur, hauteur) in zip(images, cadres, strict=True):
    # Une copie, pour ne pas dessiner le cadre dans les images de la liste.
    annotee = image.copy()
    cv2.rectangle(
      annotee, (x, y), (x + largeur, y + hauteur), color=(255, 0, 0), thickness=2
    )
    writer.write(cv2.cvtColor(annotee, cv2.COLOR_RGB2BGR))
  writer.release()


ecrire_video(images, cadres, "polo_suivi.mp4")
afficher_video("polo_suivi.mp4")

## 5. Pour aller plus loin : mettre à jour le modèle

Le cheval tourne et change d'aspect au fil de la vidéo, alors que le modèle reste celui de la première image. Écrivez `suivre_avec_mise_a_jour(images, cadre, marge=30)`, comme `suivre`, mais en remplaçant le modèle, à chaque image, par le contenu du cadre trouvé. Écrivez et affichez la vidéo de ce suivi, `polo_mise_a_jour.mp4` : le cadre suit-il mieux le cheval ? Pourquoi ?

In [ ]:
def suivre_avec_mise_a_jour(
  images: list[np.ndarray], cadre: tuple[int, int, int, int], marge: int = 30
) -> list[tuple[int, int, int, int]]:
  # Votre code ici
  ...

### Conseils supplémentaires

Une seule ligne change par rapport à `suivre` : une fois le nouveau cadre trouvé, le modèle devient son contenu dans l'image courante, en niveaux de gris. Comparez les derniers cadres des deux suivis, proches de `(189, 191, 80, 73)` sans mise à jour et de `(165, 175, 80, 73)` avec, puis regardez où se trouve le cadre dans la fin de la vidéo.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def suivre_avec_mise_a_jour(
  images: list[np.ndarray], cadre: tuple[int, int, int, int], marge: int = 30
) -> list[tuple[int, int, int, int]]:
  x, y, largeur, hauteur = cadre
  modele = color.rgb2gray(images[0][y : y + hauteur, x : x + largeur])
  cadres = [cadre]
  for image in images[1:]:
    x0, y0 = max(x - marge, 0), max(y - marge, 0)
    zone = color.rgb2gray(image[y0 : y + hauteur + marge, x0 : x + largeur + marge])
    scores = feature.match_template(zone, modele)
    dy, dx = np.unravel_index(np.argmax(scores), scores.shape)
    x, y = int(x0 + dx), int(y0 + dy)
    modele = ...
    cadres.append((x, y, largeur, hauteur))
  return cadres


cadres_mis_a_jour = suivre_avec_mise_a_jour(images, CADRE_CHEVAL)
print(cadres_mis_a_jour[::10])
ecrire_video(images, cadres_mis_a_jour, "polo_mise_a_jour.mp4")
afficher_video("polo_mise_a_jour.mp4")
```

### Solution

In [ ]:
def suivre_avec_mise_a_jour(
  images: list[np.ndarray], cadre: tuple[int, int, int, int], marge: int = 30
) -> list[tuple[int, int, int, int]]:
  x, y, largeur, hauteur = cadre
  modele = color.rgb2gray(images[0][y : y + hauteur, x : x + largeur])
  cadres = [cadre]
  for image in images[1:]:
    x0, y0 = max(x - marge, 0), max(y - marge, 0)
    zone = color.rgb2gray(image[y0 : y + hauteur + marge, x0 : x + largeur + marge])
    scores = feature.match_template(zone, modele)
    dy, dx = np.unravel_index(np.argmax(scores), scores.shape)
    x, y = int(x0 + dx), int(y0 + dy)
    # Le nouveau modèle : le contenu du cadre trouvé.
    modele = color.rgb2gray(image[y : y + hauteur, x : x + largeur])
    cadres.append((x, y, largeur, hauteur))
  return cadres


cadres_mis_a_jour = suivre_avec_mise_a_jour(images, CADRE_CHEVAL)
print(cadres_mis_a_jour[::10])
# Le cadre dérive : chaque modèle reprend la petite erreur de position du
# précédent et un peu du fond, les erreurs s'accumulent, et le cadre glisse
# peu à peu hors du cheval.
ecrire_video(images, cadres_mis_a_jour, "polo_mise_a_jour.mp4")
afficher_video("polo_mise_a_jour.mp4")

## 6. Pour aller plus loin : un algorithme de suivi d'OpenCV

OpenCV fournit des algorithmes de suivi plus élaborés, qui apprennent l'apparence de l'objet au fil de la vidéo. `cv2.TrackerMIL` (*Multiple Instance Learning*) entraîne un classifieur à distinguer l'objet de son voisinage, et le met à jour à chaque image. Écrivez `suivre_mil(images, cadre)`, qui renvoie les cadres que trouve cet algorithme, puis écrivez et affichez sa vidéo, `polo_mil.mp4`. Comparez son suivi et son temps de calcul à ceux de la corrélation.

**Documentation utile**

- https://docs.opencv.org/4.x/d0/d26/classcv_1_1TrackerMIL.html
- https://docs.opencv.org/4.x/d0/d0a/classcv_1_1Tracker.html

In [ ]:
def suivre_mil(
  images: list[np.ndarray], cadre: tuple[int, int, int, int]
) -> list[tuple[int, int, int, int]]:
  # Votre code ici
  ...

### Conseils supplémentaires

- `suivi = cv2.TrackerMIL_create()` crée l'algorithme, `suivi.init(image, cadre)` l'initialise sur la première image et son cadre, puis `ok, trouve = suivi.update(image)` donne le cadre dans chaque image suivante. `ok` vaut `False` quand l'algorithme a perdu l'objet : gardez alors le cadre précédent.
- OpenCV attend des images en BGR, et `trouve` peut contenir des flottants : convertissez-les avec `int`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def suivre_mil(
  images: list[np.ndarray], cadre: tuple[int, int, int, int]
) -> list[tuple[int, int, int, int]]:
  suivi = cv2.TrackerMIL_create()
  suivi.init(cv2.cvtColor(images[0], cv2.COLOR_RGB2BGR), cadre)
  cadres = [cadre]
  for image in images[1:]:
    ok, trouve = suivi.update(...)
    if ok:
      cadre = ...
    cadres.append(cadre)
  return cadres


%time cadres_mil = suivre_mil(images, CADRE_CHEVAL)
ecrire_video(images, cadres_mil, "polo_mil.mp4")
afficher_video("polo_mil.mp4")
```

### Solution

In [ ]:
def suivre_mil(
  images: list[np.ndarray], cadre: tuple[int, int, int, int]
) -> list[tuple[int, int, int, int]]:
  suivi = cv2.TrackerMIL_create()
  # Les images de la liste sont en RGB, OpenCV les attend en BGR.
  suivi.init(cv2.cvtColor(images[0], cv2.COLOR_RGB2BGR), cadre)
  cadres = [cadre]
  for image in images[1:]:
    ok, trouve = suivi.update(cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
    # Quand le suivi perd l'objet, le cadre reste où il était.
    if ok:
      x, y, largeur, hauteur = (int(v) for v in trouve)
      cadre = (x, y, largeur, hauteur)
    cadres.append(cadre)
  return cadres


%time cadres_mil = suivre_mil(images, CADRE_CHEVAL)
print(cadres_mil[::10])
# MIL suit lui aussi le cheval, mais bien plus lentement que la
# corrélation : il entraîne un classifieur à chaque image.
ecrire_video(images, cadres_mil, "polo_mil.mp4")
afficher_video("polo_mil.mp4")